# Colab Setup

**For Colab users:** Uncomment and run the cell below to mount Drive.  
**For local users:** Skip this cell.

In [ ]:
# Uncomment for Colab:
#from google.colab import drive
#drive.mount('/content/drive')

## Setup: Dependency Verification and Installation

This cell checks for the presence of essential Python libraries (like `numpy`, `torch`, `musdb`, `nbformat`, etc.).
If any required library is not found, it attempts to install it automatically using `pip`.
It also verifies the availability of PyTorch with CUDA, which is crucial for GPU-accelerated training.

In [ ]:
print("Verifying and installing missing packages if necessary...")

packages_to_check = [
    'numpy', 'matplotlib', 'librosa', 'tqdm', 'sklearn', 'stempeg', 'torch', 'torchvision', 'torchaudio', 'musdb', 'transformers'
]

for package in packages_to_check:
    try:
        __import__(package)
        print(f"  {package} is installed.")
    except ImportError:
        print(f"  {package} is NOT installed. Attempting to install...")
        try:
            import sys
            import subprocess
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])
            __import__(package)
            print(f"  {package} is now installed.")
        except Exception as e:
            print(f"  Failed to install {package}: {e}")

print("\n--- PyTorch CUDA status ---")
try:
    import torch
    if torch.cuda.is_available():
        print(f"  PyTorch with CUDA (version {torch.version.cuda}) is available.")
        print(f"     CUDA Device Name: {torch.cuda.get_device_name(0)}")
    else:
        print("  PyTorch is installed, but CUDA is NOT available.")
except ImportError:
    print("  PyTorch is NOT installed.")


## Imports and Environment Setup

- Import required libraries (torch, numpy, matplotlib, etc.)

- Set device (CPU/GPU)

In [ ]:
import sys
from pathlib import Path
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from IPython.display import Audio, display
import importlib
import gc

try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Final_Project_Deep_Learning')
    print(f"Colab Project Root: {PROJECT_ROOT}")
else:
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / 'mainNB.ipynb').exists():
        for p in [PROJECT_ROOT] + list(PROJECT_ROOT.parents):
            if (p / 'mainNB.ipynb').exists():
                PROJECT_ROOT = p
                break

os.chdir(PROJECT_ROOT)

DATA_DIR = PROJECT_ROOT / "data"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import models.utils as utils
importlib.reload(utils)
from models import models as ma
importlib.reload(ma)

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
set_seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"\nConfiguration Complete:")
print(f"   - Device: {device}")
print(f"   - Working Directory: {os.getcwd()}")
print(f"   - Data Directory: {DATA_DIR}")
print(f"   - Checkpoints Directory: {CHECKPOINT_DIR}")

## MUSDB18 Setup

**Quick Start:**
1. Download MUSDB18
2. Extract to project folder as `musdb18/`
3. Run preprocessing below

**Expected structure:**
```
musdb18/
  train/     (~100 songs)
  valid/     (~16 songs)
  test/      (~50 songs)
```

In [ ]:
MUSDB18_PATH = PROJECT_ROOT / "musdb18"

if not MUSDB18_PATH.exists():
    print("MUSDB18 folder not found!")
    print(f"   Expected location: {MUSDB18_PATH}")
    print("")
    print("Download MUSDB18:")
    print("   1. Register at https://zenodo.org/record/1438122")
    print("   2. Download MUSDB18-HQ.zip (~22GB)")
    print("   3. Extract to project folder as 'musdb18'")
    print("")
    MUSDB18_PATH = None
else:
    train_dir = MUSDB18_PATH / "train"
    valid_dir = MUSDB18_PATH / "valid"
    test_dir = MUSDB18_PATH / "test"

    if train_dir.exists() and test_dir.exists():
        num_train = len(list(train_dir.iterdir()))
        num_valid = len(list(valid_dir.iterdir())) if valid_dir.exists() else 0
        num_test = len(list(test_dir.iterdir()))
        print(f"MUSDB18 dataset found: {MUSDB18_PATH}")
        print(f"   Train: {num_train} tracks")
        print(f"   Valid: {num_valid} tracks")
        print(f"   Test: {num_test} tracks")
    else:
        print(f"Found musdb18 folder but missing train/test subfolders")
        print(f"   Path: {MUSDB18_PATH}")
        MUSDB18_PATH = None

mix_files_stage1 = mix_files_stage2 = tgt_files_stage1 = tgt_files_stage2 = []

## Data Preprocessing

Preprocess MUSDB18 into 8-second waveform chunks for training, validation, and testing.

In [ ]:
import shutil
import time
import zipfile
from tqdm import tqdm

skip_data_processing = True

if skip_data_processing:
    print("Data processing skipped (skip_data_processing = True)")
    print("Using pre-existing data directories...")
else:
    drive_zip_path = PROJECT_ROOT / "data.zip"

    local_extract_root = Path("/content/local_data")
    local_data_dir = local_extract_root / "data"

    FOLDERS_TO_EXTRACT = ['vocals']

    if IN_COLAB:
        print(f"Checking for local data at: {local_data_dir}")

        if not local_data_dir.exists():
            print("Data not found locally. Starting extraction...")
            print(f"Source: {drive_zip_path}")
            print(f"Destination: {local_extract_root}")
            if FOLDERS_TO_EXTRACT:
                print(f"Extracting only: {', '.join(FOLDERS_TO_EXTRACT)}")

            local_extract_root.mkdir(parents=True, exist_ok=True)

            if drive_zip_path.exists():
                t0 = time.time()
                print("Unzipping from Drive with progress tracking...")

                try:
                    with zipfile.ZipFile(drive_zip_path, 'r') as zip_ref:
                        all_files = zip_ref.namelist()
                        
                        if FOLDERS_TO_EXTRACT:
                            folders_normalized = [f.strip('/') for f in FOLDERS_TO_EXTRACT]
                            
                            file_list = [
                                f for f in all_files 
                                if any(
                                    f.startswith(f'data/{folder}/') or f.startswith(f'{folder}/')
                                    for folder in folders_normalized
                                )
                            ]
                            print(f"Filtering: {len(file_list):,} / {len(all_files):,} files selected")
                        else:
                            file_list = all_files
                            print(f"Extracting all {len(file_list):,} files")

                        for file in tqdm(file_list, desc="Extracting", unit="file"):
                            zip_ref.extract(file, local_extract_root)

                    t_final = time.time() - t0
                    print(f"Unzip complete in {t_final/60:.1f} minutes!")
                    
                    if local_data_dir.exists():
                        extracted_folders = [d.name for d in local_data_dir.iterdir() if d.is_dir()]
                        print(f"Extracted folders: {', '.join(extracted_folders)}")
                    
                    DATA_DIR = local_data_dir
                except Exception as e:
                    print(f"Unzip failed: {e}")
                    print("   Falling back to Drive path (slow)")
                    DATA_DIR = PROJECT_ROOT / "data"
            else:
                print(f"Error: Could not find {drive_zip_path}")
                print("   Please ensure 'data.zip' is uploaded to your Drive project folder.")
                DATA_DIR = PROJECT_ROOT / "data"

        else:
            print("Fast local data already exists! Skipping unzip.")
            DATA_DIR = local_data_dir

        print(f"DATA_DIR set to: {DATA_DIR}")

        total, used, free = shutil.disk_usage("/")
        print(f"Disk Space: {free // (2**30)} GB free / {total // (2**30)} GB total")

    else:
        DATA_DIR = PROJECT_ROOT / "data"
        print(f"Running locally. Using repo data: {DATA_DIR}")

    SAMPLE_RATE = 22050
    CHUNK_DURATION = 8.0
    CHUNK_OVERLAP = 4.0

    if MUSDB18_PATH:
        utils.MUSDB_SPLITS = {
            'train': MUSDB18_PATH / 'train',
            'val': MUSDB18_PATH / 'valid',
            'test': MUSDB18_PATH / 'test',
        }
        utils.DATA_DIR = DATA_DIR
        utils.SAMPLE_RATE = SAMPLE_RATE
        utils.CHUNK_DURATION = CHUNK_DURATION
        utils.CHUNK_OVERLAP = CHUNK_OVERLAP

        print(f"\n{'='*70}")
        print("CHECKING DATA FOLDERS")
        print(f"{'='*70}")
        
        has_stage1 = (DATA_DIR / 'stage1').exists()
        has_stage2 = (DATA_DIR / 'stage2').exists()
        has_vocals = (DATA_DIR / 'vocals').exists()
        
        if has_stage1:
            print("Stage1 data found - processing...")
            utils.process_stage1()
        else:
            print("Stage1 not extracted - skipping")
        
        if has_stage2:
            print("Stage2 data found - processing...")
            utils.process_stage2()
        else:
            print("Stage2 not extracted - skipping")
        
        if has_vocals:
            print("Vocals data found - processing...")
            utils.process_vocals()
        else:
            print("Vocals not extracted - skipping")

        print(f"\n{'='*70}")
        print("SYSTEM READY")
        print(f"{'='*70}")
    else:
        print("MUSDB18_PATH not set - skipping preprocessing")

## Two Architectures for Comparison

**Model (LSTM)** Sequential bidirectional LSTM with masking output.

**Model (U-Net)** 2D CNN encoder-decoder with skip connections.

In [ ]:
print("="*70)
print("MODEL A ARCHITECTURES")
print("="*70)

lstm_preview, _, _, _ = utils.initialize_model_a_lstm(device)
unet_preview, _, _, _ = utils.initialize_model_a_unet(device)

lstm_params = sum(p.numel() for p in lstm_preview.parameters())
unet_params = sum(p.numel() for p in unet_preview.parameters())

print("\nModel A1 (LSTM):")
print(f"   Parameters: {lstm_params:,}")
print(f"   Type: Bidirectional LSTM with masking")

print("\nModel A2 (U-Net):")
print(f"   Parameters: {unet_params:,}")
print(f"   Type: 2D CNN encoder-decoder")

del lstm_preview, unet_preview
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n" + "="*70)

## Train Both Models - Stage 1

Sequential training: LSTM first, then U-Net

In [ ]:
SKIP_TRAINING_STAGE1 = False
CHUNK_DURATION = 8.0

print(f"\n{'='*70}")
print('STAGE 1 TRAINING: 2 → 1')
print(f"{'='*70}\n")

ckpt_lstm_s1 = CHECKPOINT_DIR / f'model_a_lstm_stage1_{CHUNK_DURATION:.0f}s.pth'
ckpt_unet_s1 = CHECKPOINT_DIR / f'model_a_unet_stage1_{CHUNK_DURATION:.0f}s.pth'

skip_lstm_s1 = SKIP_TRAINING_STAGE1 or ckpt_lstm_s1.exists()
skip_unet_s1 = SKIP_TRAINING_STAGE1 or ckpt_unet_s1.exists()

print('Model A (LSTM) - Stage 1')
print('-' * 70)
if ckpt_lstm_s1.exists():
    print(f"LSTM Stage 1 checkpoint found: {ckpt_lstm_s1.name}")

fast_lstm_config = utils.get_training_config_lstm()
fast_lstm_config['batch_size'] = 128

model_lstm, processor_lstm, optimizer_lstm, loss_fn_lstm = utils.initialize_model_a_lstm(device)
hist_lstm_s1 = utils.train_model_stage(
    model=model_lstm,
    processor=processor_lstm,
    optimizer=optimizer_lstm,
    loss_fn=loss_fn_lstm,
    training_data_dir=DATA_DIR,
    stage='stage1',
    ckpt_path=ckpt_lstm_s1,
    device=device,
    train_config=fast_lstm_config,
    skip_training=skip_lstm_s1
)

model_lstm = model_lstm.to('cpu')
del optimizer_lstm
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

print('\nModel A (U-Net) - Stage 1')
print('-' * 70)
if ckpt_unet_s1.exists():
    print(f"U-Net Stage 1 checkpoint found: {ckpt_unet_s1.name}")

fast_unet_config = utils.get_training_config_unet()
fast_unet_config['batch_size'] = 32

print("Initializing Optimized U-Net (32 filters, 5 layers)...")
model_unet = ma.TimeFrequencyDomainUNet(
    in_channels=1,
    out_channels=1,
    base_filters=32,
    num_layers=5,
    batchnorm=True,
    dropout=0.1
).to(device)

processor_unet = utils.AudioProcessor(device=device)
optimizer_unet = optim.Adam(model_unet.parameters(), lr=fast_unet_config['learning_rate'])
loss_fn_unet = nn.MSELoss()

hist_unet_s1 = utils.train_model_stage(
    model=model_unet,
    processor=processor_unet,
    optimizer=optimizer_unet,
    loss_fn=loss_fn_unet,
    training_data_dir=DATA_DIR,
    stage='stage1',
    ckpt_path=ckpt_unet_s1,
    device=device,
    train_config=fast_unet_config,
    skip_training=skip_unet_s1
)

print(f"\n{'='*70}")
print('STAGE 1 COMPLETE!')
print(f"{'='*70}")

## Stage 1 Test Evaluation

Compute loss on held-out test set (forward pass only) to assess generalization.


In [ ]:
test_results_ckpt_lstm = CHECKPOINT_DIR / f'test_results_lstm_stage1_{CHUNK_DURATION:.0f}s.pkl'
test_results_ckpt_unet = CHECKPOINT_DIR / f'test_results_unet_stage1_{CHUNK_DURATION:.0f}s.pkl'

test_lstm_s1 = utils.load_test_results(test_results_ckpt_lstm)
test_unet_s1 = utils.load_test_results(test_results_ckpt_unet)

if test_lstm_s1 and test_unet_s1:
    print(f"Loaded cached test results from checkpoints")
    print(f"   LSTM: {test_results_ckpt_lstm.name}")
    print(f"   U-Net: {test_results_ckpt_unet.name}")
else:
    print("Running Stage 1 test evaluation (no cached results found)...\n")
    print("STAGE 1 TEST GENERALIZATION\n")

    model_lstm.to(device)
    model_unet.to(device)

    if ckpt_lstm_s1.exists():
        print(f"Loading LSTM Stage 1 weights for evaluation from: {ckpt_lstm_s1.name}")
        checkpoint = torch.load(ckpt_lstm_s1, map_location=device, weights_only=False)
        model_lstm.load_state_dict(checkpoint['model_state_dict'])
        print(f"LSTM weights loaded (trained for {checkpoint.get('epoch', '?')} epochs)")
    else:
        print("No LSTM checkpoint found - evaluating untrained model!")

    if ckpt_unet_s1.exists():
        print(f"Loading U-Net Stage 1 weights for evaluation from: {ckpt_unet_s1.name}")
        checkpoint = torch.load(ckpt_unet_s1, map_location=device, weights_only=False)
        model_unet.load_state_dict(checkpoint['model_state_dict'])
        print(f"U-Net weights loaded (trained for {checkpoint.get('epoch', '?')} epochs)")
    else:
        print("No U-Net checkpoint found - evaluating untrained model!")

    print("\n" + "="*70)
    print("EVALUATING LSTM MODEL")
    print("="*70)
    test_lstm_s1 = utils.evaluate_test_set(model_lstm, processor_lstm, DATA_DIR,
                                          'stage1', loss_fn_lstm, device)

    print("\n" + "="*70)
    print("EVALUATING U-NET MODEL")
    print("="*70)
    test_unet_s1 = utils.evaluate_test_set(model_unet, processor_unet, DATA_DIR,
                                          'stage1', loss_fn_unet, device)

    utils.save_test_results(test_lstm_s1, test_results_ckpt_lstm)
    utils.save_test_results(test_unet_s1, test_results_ckpt_unet)

if 'hist_lstm_s1' not in locals() or not hist_lstm_s1:
    ckpt_lstm = CHECKPOINT_DIR / f"model_a_lstm_stage1_{CHUNK_DURATION:.0f}s.pth"
    hist_lstm_s1 = utils.load_training_history_from_checkpoint(ckpt_lstm)

if 'hist_unet_s1' not in locals() or not hist_unet_s1:
    ckpt_unet = CHECKPOINT_DIR / f"model_a_unet_stage1_{CHUNK_DURATION:.0f}s.pth"
    hist_unet_s1 = utils.load_training_history_from_checkpoint(ckpt_unet)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

if hist_lstm_s1 and 'train_loss' in hist_lstm_s1:
    epochs_lstm = range(1, len(hist_lstm_s1['train_loss']) + 1)
    ax1.plot(epochs_lstm, hist_lstm_s1['train_loss'], 'o-', label='Train', linewidth=2)
    ax1.plot(epochs_lstm, hist_lstm_s1['val_loss'], 's--', label='Val', linewidth=2)
    
    ax1.fill_between(epochs_lstm,
                     test_lstm_s1['mean'] - test_lstm_s1['std'],
                     test_lstm_s1['mean'] + test_lstm_s1['std'],
                     alpha=0.2, color='red')

ax1.axhline(test_lstm_s1['mean'], color='red', linestyle=':', linewidth=2, label=f"Test (μ={test_lstm_s1['mean']:.4f})")
ax1.set_title('Model A (LSTM) - Stage 1', fontsize=12, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

if hist_unet_s1 and 'train_loss' in hist_unet_s1:
    epochs_unet = range(1, len(hist_unet_s1['train_loss']) + 1)
    ax2.plot(epochs_unet, hist_unet_s1['train_loss'], 'o-', label='Train', linewidth=2)
    ax2.plot(epochs_unet, hist_unet_s1['val_loss'], 's--', label='Val', linewidth=2)
    
    ax2.fill_between(epochs_unet,
                     test_unet_s1['mean'] - test_unet_s1['std'],
                     test_unet_s1['mean'] + test_unet_s1['std'],
                     alpha=0.2, color='red')

ax2.axhline(test_unet_s1['mean'], color='red', linestyle=':', linewidth=2, label=f"Test (μ={test_unet_s1['mean']:.4f})")
ax2.set_title('Model A (U-Net) - Stage 1', fontsize=12, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Train/Val/Test Comparison - Stage 1', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Stage 1 Evaluation (Spectrograms + Audio)

Compare LSTM vs U-Net separation quality on Stage 1 test samples.

In [ ]:
print("="*70)
print("STAGE 1 EVALUATION: Simplified Mixture (Vocals+Other) → Other")
print("="*70)

model_lstm = model_lstm.to(device)
model_unet = model_unet.to(device)
model_lstm.eval()
model_unet.eval()

SR = 22050
DURATION = 120.0
CHUNK_LEN = 8.0

test_base = DATA_DIR / 'stage1' / 'test'
print(f"\nSearching for Stage 1 test data at: {test_base}")

if (test_base / 'mixture').exists():
    mix_dir = test_base / 'mixture'
    print(f"Found subdirectories: mixture/ and target/")
else:
    print(f"ERROR: Stage 1 Test directory not found!")
    raise FileNotFoundError(f"Test data directory not found: {test_base}")

all_mix_files = sorted(list(mix_dir.glob('*.npy')))
if len(all_mix_files) == 0:
    raise FileNotFoundError("No .npy files found")

songs = {}
for f in all_mix_files:
    name = f.stem
    if '_chunk' in name:
        parts = name.rsplit('_chunk', 1)
        if len(parts) == 2 and parts[1].isdigit():
            song_name = parts[0]
            idx = int(parts[1])
            if song_name not in songs: songs[song_name] = {}
            songs[song_name][idx] = f
    else:
        parts = name.split('_')
        if len(parts) >= 2 and parts[-1].isdigit():
            idx = int(parts[-1])
            song_name = '_'.join(parts[:-1])
            if song_name not in songs: songs[song_name] = {}
            songs[song_name][idx] = f

song_list = sorted(songs.keys())

if len(song_list) == 1:
    selected = song_list[0]
    print(f"\nAuto-selected: {selected} (only available song)")
else:
    print(f"\nSelect a song [0-{len(song_list)-1}] or press Enter for default (0):")
    try:
        choice = input("Choice: ").strip()
        idx = int(choice) if choice else 0
    except:
        idx = 0
    selected = song_list[idx]
    print(f"Selected: {selected}")

chunks = songs[selected]

print(f"Stitching chunks...")
HOP_LENGTH = 4.0
hop_samples = int(HOP_LENGTH * SR)
target_samples = int(DURATION * SR)
mix_wav = np.zeros(target_samples, dtype=np.float32)
tgt_wav = np.zeros(target_samples, dtype=np.float32)
weights = np.zeros(target_samples, dtype=np.float32)
has_target = True

for i in sorted(chunks.keys()):
    pos = i * hop_samples
    if pos >= target_samples: break
    
    mix_chunk = np.load(chunks[i])
    tgt_file = chunks[i].name.replace('mix_', 'tgt_')
    tgt_path = test_base / 'target' / tgt_file
    
    if tgt_path.exists():
        tgt_chunk = np.load(tgt_path)
    else:
        tgt_chunk = np.zeros_like(mix_chunk)
        has_target = False
    
    valid_len = min(len(mix_chunk), target_samples - pos)
    window = np.hanning(len(mix_chunk))[:valid_len]
    
    mix_wav[pos:pos+valid_len] += mix_chunk[:valid_len] * window
    tgt_wav[pos:pos+valid_len] += tgt_chunk[:valid_len] * window
    weights[pos:pos+valid_len] += window

mix_wav = np.divide(mix_wav, weights, where=weights > 0)
tgt_wav = np.divide(tgt_wav, weights, where=weights > 0)

print("\nRunning Stage 1 inference...")
est_lstm_s1 = utils.sliding_window_inference(model_lstm, processor_lstm, mix_wav, chunk_len=CHUNK_LEN, sr=SR, device=device)
est_unet_s1 = utils.sliding_window_inference(model_unet, processor_unet, mix_wav, chunk_len=CHUNK_LEN, sr=SR, device=device)

print(f"\n{'='*70}\nSTAGE 1 RESULTS\n{'='*70}")

fig, axes = plt.subplots(3, 2, figsize=(15, 12))

axes[0,0].imshow(utils.to_spec(mix_wav, processor_lstm), aspect='auto', origin='lower', cmap='viridis')
axes[0,0].set_title("Input: Simplified Mixture (Vocals+Other)", fontweight='bold')

if has_target:
    axes[0,1].imshow(utils.to_spec(tgt_wav, processor_lstm), aspect='auto', origin='lower', cmap='viridis')
    axes[0,1].set_title("Ground Truth: Other", fontweight='bold')
else:
    axes[0,1].text(0.5, 0.5, "Target Not Available", ha='center', va='center', transform=axes[0,1].transAxes)

spec_tgt_lstm = utils.to_spec(tgt_wav, processor_lstm)
spec_pred_lstm = utils.to_spec(est_lstm_s1, processor_lstm)
min_len = min(spec_tgt_lstm.shape[1], spec_pred_lstm.shape[1])
err_lstm = np.abs(spec_pred_lstm[:, :min_len] - spec_tgt_lstm[:, :min_len])

axes[1,0].imshow(err_lstm, aspect='auto', origin='lower', cmap='magma')
axes[1,0].set_title("LSTM Error Map (|Pred - Tgt|)", fontweight='bold')

axes[1,1].imshow(spec_pred_lstm, aspect='auto', origin='lower', cmap='viridis')
axes[1,1].set_title("LSTM Prediction (Stage 1)", fontweight='bold')

spec_tgt_unet = utils.to_spec(tgt_wav, processor_unet)
spec_pred_unet = utils.to_spec(est_unet_s1, processor_unet)
min_len_u = min(spec_tgt_unet.shape[1], spec_pred_unet.shape[1])
err_unet = np.abs(spec_pred_unet[:, :min_len_u] - spec_tgt_unet[:, :min_len_u])

axes[2,0].imshow(err_unet, aspect='auto', origin='lower', cmap='magma')
axes[2,0].set_title("U-Net Error Map (|Pred - Tgt|)", fontweight='bold')

axes[2,1].imshow(spec_pred_unet, aspect='auto', origin='lower', cmap='viridis')
axes[2,1].set_title("U-Net Prediction (Stage 1)", fontweight='bold')

for ax in axes.flatten():
    ax.set_xlabel("Time")
    ax.set_ylabel("Frequency")

plt.suptitle(f"Stage 1 Evaluation: {selected}", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nAudio Playback:")
print("Input (Mix):")
display(Audio(mix_wav, rate=SR))

print("\nGround Truth (Target):")
display(Audio(tgt_wav, rate=SR))

print("\nLSTM Prediction:")
display(Audio(est_lstm_s1, rate=SR))

print("\nU-Net Prediction:")
display(Audio(est_unet_s1, rate=SR))

## Train Both Models - Stage 2

Curriculum step: Stage 2 uses 4→1 channels and continues from Stage 1 weights.

In [ ]:
STAGE2_ENABLED = True
SKIP_TRAINING_STAGE2 = False

FAST_LSTM_CONFIG = utils.get_training_config_lstm()
FAST_LSTM_CONFIG['batch_size'] = 128

FAST_UNET_CONFIG = utils.get_training_config_unet()
FAST_UNET_CONFIG['batch_size'] = 32

ckpt_lstm_s2 = CHECKPOINT_DIR / f"model_a_lstm_stage2_{CHUNK_DURATION:.0f}s.pth"
ckpt_unet_s2 = CHECKPOINT_DIR / f"model_a_unet_stage2_{CHUNK_DURATION:.0f}s.pth"
ckpt_lstm_s1 = CHECKPOINT_DIR / f"model_a_lstm_stage1_{CHUNK_DURATION:.0f}s.pth"
ckpt_unet_s1 = CHECKPOINT_DIR / f"model_a_unet_stage1_{CHUNK_DURATION:.0f}s.pth"

hist_lstm_s2 = {}
hist_unet_s2 = {}

if not STAGE2_ENABLED:
    print("Stage 2 training disabled (STAGE2_ENABLED = False)")
else:
    print("1. Model A (LSTM) - Stage 2")
    print("-" * 70)

    model_lstm, processor_lstm, optimizer_lstm, loss_fn_lstm = utils.initialize_model_a_lstm(device)

    if ckpt_lstm_s1.exists():
        print(f"Loading Stage 2 weights from: {ckpt_lstm_s1.name}")
        checkpoint = torch.load(ckpt_lstm_s1, map_location=device)
        model_lstm.load_state_dict(checkpoint['model_state_dict'])
        print(f"LSTM Stage 2 weights loaded successfully.")
    else:
        print("LSTM Stage 2 checkpoint NOT found. Starting from scratch (not ideal).")

    hist_lstm_s2 = utils.train_model_stage(
        model=model_lstm,
        processor=processor_lstm,
        optimizer=optimizer_lstm,
        loss_fn=loss_fn_lstm,
        training_data_dir=DATA_DIR,
        stage="stage2",
        ckpt_path=ckpt_lstm_s2,
        device=device,
        train_config=FAST_LSTM_CONFIG,
        skip_training=SKIP_TRAINING_STAGE2
    )

    model_lstm = model_lstm.to('cpu')
    del optimizer_lstm
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    print("\n2. Model A (U-Net) - Stage 2")
    print("-" * 70)

    model_unet = ma.TimeFrequencyDomainUNet(
        in_channels=1,
        out_channels=1,
        base_filters=32,
        num_layers=5,
        batchnorm=True,
        dropout=0.1
    ).to(device)

    processor_unet = utils.AudioProcessor(device=device)
    optimizer_unet = optim.Adam(model_unet.parameters(), lr=FAST_UNET_CONFIG['learning_rate'])
    loss_fn_unet = nn.MSELoss()

    if ckpt_unet_s1.exists():
        print(f"Loading Stage 2 weights from: {ckpt_unet_s1.name}")
        checkpoint = torch.load(ckpt_unet_s1, map_location=device)
        try:
            model_unet.load_state_dict(checkpoint['model_state_dict'])
            print(f"U-Net Stage 2 weights loaded successfully.")
        except RuntimeError as e:
            print(f"CRITICAL ERROR loading weights: {e}")
            print("   (This usually means the model structure in Stage 2 doesn't match Stage 1)")
            raise e
    else:
        print("U-Net Stage 2 checkpoint NOT found. Starting from scratch.")

    hist_unet_s2 = utils.train_model_stage(
        model=model_unet,
        processor=processor_unet,
        optimizer=optimizer_unet,
        loss_fn=loss_fn_unet,
        training_data_dir=DATA_DIR,
        stage="stage2",
        ckpt_path=ckpt_unet_s2,
        device=device,
        train_config=FAST_UNET_CONFIG,
        skip_training=SKIP_TRAINING_STAGE2
    )

print(f"\n{'='*70}")
print("STAGE 2 COMPLETE!")
print(f"{'='*70}")

## Stage 2 Test Evaluation

Compute loss on held-out test set (forward pass only) to assess generalization.


In [ ]:
test_results_ckpt_lstm_s2 = CHECKPOINT_DIR / f'test_results_lstm_stage2_{CHUNK_DURATION:.0f}s.pkl'
test_results_ckpt_unet_s2 = CHECKPOINT_DIR / f'test_results_unet_stage2_{CHUNK_DURATION:.0f}s.pkl'

test_lstm_s2 = utils.load_test_results(test_results_ckpt_lstm_s2)
test_unet_s2 = utils.load_test_results(test_results_ckpt_unet_s2)

if test_lstm_s2 and test_unet_s2:
    print(f"Loaded cached test results from checkpoints")
    print(f"   LSTM: {test_results_ckpt_lstm_s2.name}")
    print(f"   U-Net: {test_results_ckpt_unet_s2.name}")
else:
    print("Running Stage 2 test evaluation (no cached results found)...\n")
    print("STAGE 2 TEST GENERALIZATION\n")

    model_lstm_s2, processor_lstm_s2, _, loss_fn_lstm_s2 = utils.initialize_model_a_lstm(device)

    model_unet_s2 = ma.TimeFrequencyDomainUNet(
        in_channels=1,
        out_channels=1,
        base_filters=32,
        num_layers=5,
        batchnorm=True,
        dropout=0.1
    ).to(device)
    processor_unet_s2 = utils.AudioProcessor(device=device)
    loss_fn_unet_s2 = torch.nn.MSELoss()

    if ckpt_lstm_s2.exists():
        print(f"Loading LSTM Stage 2 weights from: {ckpt_lstm_s2.name}")
        checkpoint = torch.load(ckpt_lstm_s2, map_location=device, weights_only=False)
        model_lstm_s2.load_state_dict(checkpoint['model_state_dict'])
        print(f"LSTM weights loaded (trained for {checkpoint.get('epoch', '?')} epochs)")
    else:
        print("No LSTM Stage 2 checkpoint found - evaluating untrained model!")

    if ckpt_unet_s2.exists():
        print(f"Loading U-Net Stage 2 weights from: {ckpt_unet_s2.name}")
        checkpoint = torch.load(ckpt_unet_s2, map_location=device, weights_only=False)
        model_unet_s2.load_state_dict(checkpoint['model_state_dict'])
        print(f"U-Net weights loaded (trained for {checkpoint.get('epoch', '?')} epochs)")
    else:
        print("No U-Net Stage 2 checkpoint found - evaluating untrained model!")

    print("\n" + "="*70)
    print("EVALUATING LSTM MODEL (STAGE 2)")
    print("="*70)
    test_lstm_s2 = utils.evaluate_test_set(model_lstm_s2, processor_lstm_s2, DATA_DIR,
                                          'stage2', loss_fn_lstm_s2, device)

    print("\n" + "="*70)
    print("EVALUATING U-NET MODEL (STAGE 2)")
    print("="*70)
    test_unet_s2 = utils.evaluate_test_set(model_unet_s2, processor_unet_s2, DATA_DIR,
                                          'stage2', loss_fn_unet_s2, device)

    utils.save_test_results(test_lstm_s2, test_results_ckpt_lstm_s2)
    utils.save_test_results(test_unet_s2, test_results_ckpt_unet_s2)

if not hist_lstm_s2:
    ckpt_lstm_s2 = CHECKPOINT_DIR / f"model_a_lstm_stage2_{CHUNK_DURATION:.0f}s.pth"
    hist_lstm_s2 = utils.load_training_history_from_checkpoint(ckpt_lstm_s2)

if not hist_unet_s2:
    ckpt_unet_s2 = CHECKPOINT_DIR / f"model_a_unet_stage2_{CHUNK_DURATION:.0f}s.pth"
    hist_unet_s2 = utils.load_training_history_from_checkpoint(ckpt_unet_s2)

if not STAGE2_ENABLED:
    print("Stage 2 plotting skipped (STAGE2_ENABLED = False)")
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    if hist_lstm_s2 and 'train_loss' in hist_lstm_s2:
        epochs_lstm = range(1, len(hist_lstm_s2['train_loss']) + 1)
        ax1.plot(epochs_lstm, hist_lstm_s2['train_loss'], 'o-', label='Train', linewidth=2)
        ax1.plot(epochs_lstm, hist_lstm_s2['val_loss'], 's--', label='Val', linewidth=2)
        
        ax1.fill_between(epochs_lstm,
                         test_lstm_s2['mean'] - test_lstm_s2['std'],
                         test_lstm_s2['mean'] + test_lstm_s2['std'],
                         alpha=0.2, color='red')

    ax1.axhline(test_lstm_s2['mean'], color='red', linestyle=':', linewidth=2, label=f"Test (μ={test_lstm_s2['mean']:.4f})")
    ax1.set_title('Model A (LSTM) - Stage 2', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    if hist_unet_s2 and 'train_loss' in hist_unet_s2:
        epochs_unet = range(1, len(hist_unet_s2['train_loss']) + 1)
        ax2.plot(epochs_unet, hist_unet_s2['train_loss'], 'o-', label='Train', linewidth=2)
        ax2.plot(epochs_unet, hist_unet_s2['val_loss'], 's--', label='Val', linewidth=2)
        
        ax2.fill_between(epochs_unet,
                         test_unet_s2['mean'] - test_unet_s2['std'],
                         test_unet_s2['mean'] + test_unet_s2['std'],
                         alpha=0.2, color='red')

    ax2.axhline(test_unet_s2['mean'], color='red', linestyle=':', linewidth=2, label=f"Test (μ={test_unet_s2['mean']:.4f})")
    ax2.set_title('Model A (U-Net) - Stage 2', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.suptitle('Train/Val/Test Comparison - Stage 2 (Full Mix -> Vocals)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## Stage 2 Evaluation (Spectrograms + Audio)

Compare LSTM vs U-Net separation quality on Stage 2 test samples.

In [ ]:
print("="*70)
print("STAGE 2 EVALUATION: Full Band Mixture → Vocals")
print("="*70)

model_lstm = model_lstm.to(device)
model_unet = model_unet.to(device)
model_lstm.eval()
model_unet.eval()

SR = 22050
DURATION = 120.0
CHUNK_LEN = 8.0

test_base = DATA_DIR / 'stage2' / 'test'
print(f"\nSearching for Stage 2 test data at: {test_base}")

if (test_base / 'mixture').exists():
    mix_dir = test_base / 'mixture'
    print(f"Found subdirectories: mixture/ and target/")
else:
    print(f"ERROR: Stage 2 Test directory not found!")
    raise FileNotFoundError(f"Test data directory not found: {test_base}")

all_mix_files = sorted(list(mix_dir.glob('*.npy')))
if len(all_mix_files) == 0:
    raise FileNotFoundError("No .npy files found")

songs = {}
for f in all_mix_files:
    name = f.stem
    if '_chunk' in name:
        parts = name.rsplit('_chunk', 1)
        if len(parts) == 2 and parts[1].isdigit():
            song_name = parts[0]
            idx = int(parts[1])
            if song_name not in songs: songs[song_name] = {}
            songs[song_name][idx] = f
    else:
        parts = name.split('_')
        if len(parts) >= 2 and parts[-1].isdigit():
            idx = int(parts[-1])
            song_name = '_'.join(parts[:-1])
            if song_name not in songs: songs[song_name] = {}
            songs[song_name][idx] = f

song_list = sorted(songs.keys())

if len(song_list) == 1:
    selected = song_list[0]
    print(f"\nAuto-selected: {selected} (only available song)")
else:
    print(f"\nSelect a song [0-{len(song_list)-1}] or press Enter for default (0):")
    try:
        choice = input("Choice: ").strip()
        idx = int(choice) if choice else 0
    except:
        idx = 0
    selected = song_list[idx]
    print(f"Selected: {selected}")

chunks = songs[selected]

print(f"Stitching chunks...")
HOP_LENGTH = 4.0
hop_samples = int(HOP_LENGTH * SR)
target_samples = int(DURATION * SR)
mix_wav = np.zeros(target_samples, dtype=np.float32)
tgt_wav = np.zeros(target_samples, dtype=np.float32)
weights = np.zeros(target_samples, dtype=np.float32)
has_target = True

for i in sorted(chunks.keys()):
    pos = i * hop_samples
    if pos >= target_samples: break
    
    mix_chunk = np.load(chunks[i])
    tgt_file = chunks[i].name.replace('mix_', 'tgt_')
    tgt_path = test_base / 'target' / tgt_file
    
    if tgt_path.exists():
        tgt_chunk = np.load(tgt_path)
    else:
        tgt_chunk = np.zeros_like(mix_chunk)
        has_target = False
    
    valid_len = min(len(mix_chunk), target_samples - pos)
    window = np.hanning(len(mix_chunk))[:valid_len]
    
    mix_wav[pos:pos+valid_len] += mix_chunk[:valid_len] * window
    tgt_wav[pos:pos+valid_len] += tgt_chunk[:valid_len] * window
    weights[pos:pos+valid_len] += window

mix_wav = np.divide(mix_wav, weights, where=weights > 0)
tgt_wav = np.divide(tgt_wav, weights, where=weights > 0)

print("\nRunning Stage 2 inference...")
est_lstm_s2 = utils.sliding_window_inference(model_lstm, processor_lstm, mix_wav, chunk_len=CHUNK_LEN, sr=SR, device=device)
est_unet_s2 = utils.sliding_window_inference(model_unet, processor_unet, mix_wav, chunk_len=CHUNK_LEN, sr=SR, device=device)

print(f"\n{'='*70}\nSTAGE 2 RESULTS\n{'='*70}")

fig, axes = plt.subplots(3, 2, figsize=(15, 12))

axes[0,0].imshow(utils.to_spec(mix_wav, processor_lstm), aspect='auto', origin='lower', cmap='viridis')
axes[0,0].set_title("Input: Full Band Mixture", fontweight='bold')

if has_target:
    axes[0,1].imshow(utils.to_spec(tgt_wav, processor_lstm), aspect='auto', origin='lower', cmap='viridis')
    axes[0,1].set_title("Ground Truth: Vocals", fontweight='bold')
else:
    axes[0,1].text(0.5, 0.5, "Target Not Available", ha='center')

spec_tgt_lstm = utils.to_spec(tgt_wav, processor_lstm)
spec_pred_lstm = utils.to_spec(est_lstm_s2, processor_lstm)
min_len = min(spec_tgt_lstm.shape[1], spec_pred_lstm.shape[1])
err_lstm = np.abs(spec_pred_lstm[:, :min_len] - spec_tgt_lstm[:, :min_len])

axes[1,0].imshow(err_lstm, aspect='auto', origin='lower', cmap='magma')
axes[1,0].set_title("LSTM Error Map (|Pred - Tgt|)", fontweight='bold')

axes[1,1].imshow(spec_pred_lstm, aspect='auto', origin='lower', cmap='viridis')
axes[1,1].set_title("LSTM Prediction (Stage 2)", fontweight='bold')

spec_tgt_unet = utils.to_spec(tgt_wav, processor_unet)
spec_pred_unet = utils.to_spec(est_unet_s2, processor_unet)
min_len_u = min(spec_tgt_unet.shape[1], spec_pred_unet.shape[1])
err_unet = np.abs(spec_pred_unet[:, :min_len_u] - spec_tgt_unet[:, :min_len_u])

axes[2,0].imshow(err_unet, aspect='auto', origin='lower', cmap='magma')
axes[2,0].set_title("U-Net Error Map (|Pred - Tgt|)", fontweight='bold')

axes[2,1].imshow(spec_pred_unet, aspect='auto', origin='lower', cmap='viridis')
axes[2,1].set_title("U-Net Prediction (Stage 2)", fontweight='bold')

plt.tight_layout()
plt.show()

print("\nAudio Playback:")
print("Input (Mix):")
display(Audio(mix_wav, rate=SR))

print("\nGround Truth (Target):")
display(Audio(tgt_wav, rate=SR))

print("\nLSTM Prediction:")
display(Audio(est_lstm_s2, rate=SR))

print("\nU-Net Prediction:")
display(Audio(est_unet_s2, rate=SR))

## Quantitative Evaluation

Compute BSS metrics (SDR/SIR/SAR) on test set using museval library.

In [ ]:
NUM_TEST_SAMPLES = 500
STAGE_FOR_EVAL = "stage2"
RANDOM_SAMPLE_TEST_CHUNKS = True
RANDOM_SEED = 42
FORCE_RECOMPUTE_METRICS = False

metrics_ckpt = CHECKPOINT_DIR / f"quant_metrics_unet_vs_lstm{STAGE_FOR_EVAL}_{NUM_TEST_SAMPLES}samples.pkl"
print(f"Metrics checkpoint: {metrics_ckpt}")

if FORCE_RECOMPUTE_METRICS and metrics_ckpt.exists():
    metrics_ckpt.unlink()
    print("Removed existing metrics checkpoint (force recompute enabled).")

metrics = utils.evaluate_separation_quality(
    model_1=model_lstm,
    model_2=model_unet,
    processor_1=processor_lstm,
    processor_2=processor_unet,
    test_data_dir=DATA_DIR,
    stage=STAGE_FOR_EVAL,
    num_samples=NUM_TEST_SAMPLES,
    sr=22050,
    device=device,
    save_path=metrics_ckpt,
    load_if_exists=True,
    random_sampling=RANDOM_SAMPLE_TEST_CHUNKS,
    random_seed=RANDOM_SEED
)

## Custom Song Inference (Upload)

Upload a song (or place it in the folder below) and run inference with both models. This is the final step.

In [ ]:
UPLOAD_DIR = DATA_DIR / "user_uploads"
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    try:
        from google.colab import files
        uploaded = files.upload()
        for name, data in uploaded.items():
            out_path = UPLOAD_DIR / name
            with open(out_path, "wb") as f:
                f.write(data)
            print(f"Saved: {out_path}")
    except Exception as e:
        print(f"Colab upload failed: {e}")
        print(f"Place a file manually in: {UPLOAD_DIR}")
else:
    print(f"Place your audio file in: {UPLOAD_DIR}")

exts = ["*.wav", "*.mp3", "*.flac", "*.ogg", "*.m4a"]
audio_files = []
for ext in exts:
    audio_files += list(UPLOAD_DIR.glob(ext))

audio_files = sorted(audio_files, key=lambda p: p.stat().st_mtime, reverse=True)

if not audio_files:
    print("No audio files found in upload folder.")
else:
    audio_path = audio_files[0]
    print(f"Using file: {audio_path.name}")

    utils.compare_models_on_audio_file(
        file_path=audio_path,
        model_lstm=model_lstm,
        model_unet=model_unet,
        processor_lstm=processor_lstm,
        processor_unet=processor_unet,
        device=device,
        sr=22050,
        duration=None,
        unet_batch_size=16
    )


# Adding Attention To our Unet
In this part we will try to improve seperation by adding attention to our Unet

In [ ]:
print("="*70)
print("MODEL B ARCHITECTURE: UNetAttention (U-Net + Self-Attention)")
print("="*70)

try:
    unettention_preview = ma.UNetAttention(
        in_channels=1,
        out_channels=1,
        base_filters=32,
        num_layers=5,
        num_heads=4
    ).to(device)

    attn_params = sum(p.numel() for p in unettention_preview.parameters())

    print("\nModel B (Unettention):")
    print(f"   Parameters: {attn_params:,}")
    print(f"   Type: U-Net with Multi-Head Self-Attention at bottleneck")
    print(f"   Heads: 4 | Layers: 5 | Base Filters: 32")

    del unettention_preview
    import gc
    gc.collect()
    torch.cuda.empty_cache()

except AttributeError:
    print("Error: 'Unettention' class not found in models.py.")
    print("   Please ensure you added the class code from the previous step!")

print("\n" + "="*70)

## Train Unettention Model - Stage 1

In [ ]:
print(f"\n{'='*70}")
print('STAGE 1 TRAINING: UNetAttention (2 → 1)')
print(f"{'='*70}\n")

ckpt_attn_s1 = CHECKPOINT_DIR / f'model_unetattention_stage1_{CHUNK_DURATION:.0f}s.pth'
skip_attn_s1 = False

attn_config = utils.get_training_config_unet()
attn_config['batch_size'] = 32

print("Initializing UNetAttention (32 filters, 4 layers, 4 heads)...")
model_attn_s1 = ma.UNetAttention(
    in_channels=1,
    out_channels=1,
    base_filters=32,
    num_layers=4,
    num_heads=4,
    batchnorm=True,
    dropout=0.1
).to(device)

processor_attn = utils.AudioProcessor(device=device)
optimizer_attn = optim.Adam(model_attn_s1.parameters(), lr=attn_config['learning_rate'])
loss_fn_attn = nn.MSELoss()

hist_attn_s1 = utils.train_model_stage(
    model=model_attn_s1,
    processor=processor_attn,
    optimizer=optimizer_attn,
    loss_fn=loss_fn_attn,
    training_data_dir=DATA_DIR,
    stage='stage1',
    ckpt_path=ckpt_attn_s1,
    device=device,
    train_config=attn_config,
    skip_training=ckpt_attn_s1.exists() and not skip_attn_s1
)

## Compare Training Results (Stage 1)

Side-by-side comparison of Standard U-Net vs. UNetAttention training curves. We look for faster convergence or lower final validation loss to see if the attention mechanism helps.

In [ ]:
if 'hist_unet_s1' not in locals() or not hist_unet_s1:
    ckpt_unet_s1 = CHECKPOINT_DIR / f"model_a_unet_stage1_{CHUNK_DURATION:.0f}s.pth"
    hist_unet_s1 = utils.load_training_history_from_checkpoint(ckpt_unet_s1)

if 'hist_attn_s1' not in locals() or not hist_attn_s1:
    ckpt_attn_s1 = CHECKPOINT_DIR / f"model_unetattention_stage1_{CHUNK_DURATION:.0f}s.pth"
    hist_attn_s1 = utils.load_training_history_from_checkpoint(ckpt_attn_s1)

test_results_ckpt_unet_s1 = CHECKPOINT_DIR / f"test_results_unet_stage1_{CHUNK_DURATION:.0f}s.pkl"
test_results_ckpt_attn_s1 = CHECKPOINT_DIR / f"test_results_unetattention_stage1_{CHUNK_DURATION:.0f}s.pkl"

test_unet_s1 = utils.load_test_results(test_results_ckpt_unet_s1)
test_attn_s1 = utils.load_test_results(test_results_ckpt_attn_s1)

if hist_unet_s1 and hist_attn_s1:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    if 'train_loss' in hist_unet_s1:
        epochs_unet = range(1, len(hist_unet_s1['train_loss']) + 1)
        ax1.plot(epochs_unet, hist_unet_s1['train_loss'], 'o-', label='Train', linewidth=2)
        ax1.plot(epochs_unet, hist_unet_s1['val_loss'], 's--', label='Val', linewidth=2)

        if test_unet_s1 and 'mean' in test_unet_s1 and 'std' in test_unet_s1:
            ax1.fill_between(
                epochs_unet,
                test_unet_s1['mean'] - test_unet_s1['std'],
                test_unet_s1['mean'] + test_unet_s1['std'],
                alpha=0.2, color='red'
            )
            ax1.axhline(
                test_unet_s1['mean'],
                color='red', linestyle=':', linewidth=2,
                label=f"Test (μ={test_unet_s1['mean']:.4f})"
            )

    ax1.set_title('Standard U-Net - Stage 1', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    if 'train_loss' in hist_attn_s1:
        epochs_attn = range(1, len(hist_attn_s1['train_loss']) + 1)
        ax2.plot(epochs_attn, hist_attn_s1['train_loss'], 'o-', label='Train', linewidth=2)
        ax2.plot(epochs_attn, hist_attn_s1['val_loss'], 's--', label='Val', linewidth=2)

        if test_attn_s1 and 'mean' in test_attn_s1 and 'std' in test_attn_s1:
            ax2.fill_between(
                epochs_attn,
                test_attn_s1['mean'] - test_attn_s1['std'],
                test_attn_s1['mean'] + test_attn_s1['std'],
                alpha=0.2, color='red'
            )
            ax2.axhline(
                test_attn_s1['mean'],
                color='red', linestyle=':', linewidth=2,
                label=f"Test (μ={test_attn_s1['mean']:.4f})"
            )

    ax2.set_title('UNetAttention - Stage 1', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.suptitle('Train/Val/Test Comparison - Stage 1 (U-Net vs UNetAttention)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Could not plot comparison - missing training history for one or both models.")

## Stage 1 Visual Evaluation (Spectrograms + Audio)

Visual inspection of the Attention model's separation quality on a test sample. We look for clearer separation in the spectrograms compared to previous models.

In [ ]:
print("="*70)
print("STAGE 1 EVALUATION: U-Net vs UNetAttention (Simplified Mixture → Other)")
print("="*70)

model_unet = model_unet.to(device)
model_attn_s1 = model_attn_s1.to(device)
model_unet.eval()
model_attn_s1.eval()

SR = 22050
DURATION = 120.0
CHUNK_LEN = 8.0

test_base = DATA_DIR / 'stage1' / 'test'
print(f"\nSearching for Stage 1 test data at: {test_base}")

if (test_base / 'mixture').exists():
    mix_dir = test_base / 'mixture'
    print("Found subdirectories: mixture/ and target/")
else:
    print("ERROR: Stage 1 Test directory not found!")
    raise FileNotFoundError(f"Test data directory not found: {test_base}")

all_mix_files = sorted(list(mix_dir.glob('*.npy')))
if len(all_mix_files) == 0:
    raise FileNotFoundError("No .npy files found")

songs = {}
for f in all_mix_files:
    name = f.stem
    if '_chunk' in name:
        parts = name.rsplit('_chunk', 1)
        if len(parts) == 2 and parts[1].isdigit():
            song_name = parts[0]
            idx = int(parts[1])
            if song_name not in songs:
                songs[song_name] = {}
            songs[song_name][idx] = f
    else:
        parts = name.split('_')
        if len(parts) >= 2 and parts[-1].isdigit():
            idx = int(parts[-1])
            song_name = '_'.join(parts[:-1])
            if song_name not in songs:
                songs[song_name] = {}
            songs[song_name][idx] = f

song_list = sorted(songs.keys())

if len(song_list) == 1:
    selected = song_list[0]
    print(f"\nAuto-selected: {selected} (only available song)")
else:
    print(f"\nSelect a song [0-{len(song_list)-1}] or press Enter for default (0):")
    try:
        choice = input("Choice: ").strip()
        idx = int(choice) if choice else 0
    except:
        idx = 0
    selected = song_list[idx]
    print(f"Selected: {selected}")

chunks = songs[selected]

print("Stitching chunks...")
HOP_LENGTH = 4.0
hop_samples = int(HOP_LENGTH * SR)
target_samples = int(DURATION * SR)
mix_wav = np.zeros(target_samples, dtype=np.float32)
tgt_wav = np.zeros(target_samples, dtype=np.float32)
weights = np.zeros(target_samples, dtype=np.float32)
has_target = True

for i in sorted(chunks.keys()):
    pos = i * hop_samples
    if pos >= target_samples:
        break

    mix_chunk = np.load(chunks[i])
    tgt_file = chunks[i].name.replace('mix_', 'tgt_')
    tgt_path = test_base / 'target' / tgt_file

    if tgt_path.exists():
        tgt_chunk = np.load(tgt_path)
    else:
        tgt_chunk = np.zeros_like(mix_chunk)
        has_target = False

    valid_len = min(len(mix_chunk), target_samples - pos)
    window = np.hanning(len(mix_chunk))[:valid_len]

    mix_wav[pos:pos+valid_len] += mix_chunk[:valid_len] * window
    tgt_wav[pos:pos+valid_len] += tgt_chunk[:valid_len] * window
    weights[pos:pos+valid_len] += window

mix_wav = np.divide(mix_wav, weights, where=weights > 0)
tgt_wav = np.divide(tgt_wav, weights, where=weights > 0)

print(f"Stitching complete: mix_wav shape={mix_wav.shape}, range=[{mix_wav.min():.4f}, {mix_wav.max():.4f}]")
print(f"Stitching complete: tgt_wav shape={tgt_wav.shape}, range=[{tgt_wav.min():.4f}, {tgt_wav.max():.4f}]")

print("\nRunning Stage 1 inference...")
est_unet = utils.sliding_window_inference(
    model_unet, processor_unet, mix_wav, chunk_len=CHUNK_LEN, sr=SR, device=device
)
est_attn = utils.sliding_window_inference(
    model_attn_s1, processor_attn, mix_wav, chunk_len=CHUNK_LEN, sr=SR, device=device
)

print(f"\n{'='*70}\nSTAGE 1 RESULTS: U-Net vs UNetAttention\n{'='*70}")

fig, axes = plt.subplots(3, 2, figsize=(15, 12))

axes[0,0].imshow(utils.to_spec(mix_wav, processor_unet), aspect='auto', origin='lower', cmap='viridis')
axes[0,0].set_title("Input: Simplified Mixture", fontweight='bold')

if has_target:
    axes[0,1].imshow(utils.to_spec(tgt_wav, processor_unet), aspect='auto', origin='lower', cmap='viridis')
    axes[0,1].set_title("Ground Truth: Other", fontweight='bold')
else:
    axes[0,1].text(0.5, 0.5, "Target Not Available", ha='center', va='center', transform=axes[0,1].transAxes)

spec_tgt_unet = utils.to_spec(tgt_wav, processor_unet)
spec_pred_unet = utils.to_spec(est_unet, processor_unet)
min_len_u = min(spec_tgt_unet.shape[1], spec_pred_unet.shape[1])
err_unet = np.abs(spec_pred_unet[:, :min_len_u] - spec_tgt_unet[:, :min_len_u])

axes[1,0].imshow(err_unet, aspect='auto', origin='lower', cmap='magma')
axes[1,0].set_title("U-Net Error Map (|Pred - Tgt|)", fontweight='bold')

axes[1,1].imshow(spec_pred_unet, aspect='auto', origin='lower', cmap='viridis')
axes[1,1].set_title("U-Net Prediction (Stage 1)", fontweight='bold')

spec_tgt_attn = utils.to_spec(tgt_wav, processor_attn)
spec_pred_attn = utils.to_spec(est_attn, processor_attn)
min_len_a = min(spec_tgt_attn.shape[1], spec_pred_attn.shape[1])
err_attn = np.abs(spec_pred_attn[:, :min_len_a] - spec_tgt_attn[:, :min_len_a])

axes[2,0].imshow(err_attn, aspect='auto', origin='lower', cmap='magma')
axes[2,0].set_title("UNetAttention Error Map (|Pred - Tgt|)", fontweight='bold')

axes[2,1].imshow(spec_pred_attn, aspect='auto', origin='lower', cmap='viridis')
axes[2,1].set_title("UNetAttention Prediction (Stage 1)", fontweight='bold')

for ax in axes.flatten():
    ax.set_xlabel("Time")
    ax.set_ylabel("Frequency")

plt.suptitle(f"Stage 1 Evaluation: {selected}", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nAudio Playback:")
print("Input (Mix):")
display(Audio(mix_wav, rate=SR))

if has_target:
    print("\nGround Truth (Target):")
    display(Audio(tgt_wav, rate=SR))

print("\nU-Net Prediction:")
display(Audio(est_unet, rate=SR))

print("\nUNetAttention Prediction:")
display(Audio(est_attn, rate=SR))

## Train UNetAttention - Stage 2

In [ ]:
print(f"\n{'='*70}")
print('STAGE 2 TRAINING: UNetAttention (Full Mix -> Target)')
print(f"{'='*70}\n")

ckpt_attn_s2 = CHECKPOINT_DIR / f'model_unetattention_stage2_{CHUNK_DURATION:.0f}s.pth'
skip_attn_s2 = False

model_attn_s2 = ma.UNetAttention(
    in_channels=1,
    out_channels=1,
    base_filters=32,
    num_layers=4,
    num_heads=4,
    batchnorm=True,
    dropout=0.1
).to(device)

if ckpt_attn_s1.exists():
    print(f"Loading Stage 1 weights from: {ckpt_attn_s1.name}")
    checkpoint = torch.load(ckpt_attn_s1, map_location=device)
    model_attn_s2.load_state_dict(checkpoint['model_state_dict'])
    print("Curriculum Learning: Stage 1 weights loaded.")
else:
    print("Stage 1 checkpoint not found! Training from scratch (harder convergence).")

processor_attn_s2 = utils.AudioProcessor(device=device)
optimizer_attn_s2 = optim.Adam(model_attn_s2.parameters(), lr=attn_config['learning_rate'])
loss_fn_attn_s2 = nn.MSELoss()

hist_attn_s2 = utils.train_model_stage(
    model=model_attn_s2,
    processor=processor_attn_s2,
    optimizer=optimizer_attn_s2,
    loss_fn=loss_fn_attn_s2,
    training_data_dir=DATA_DIR,
    stage='stage2',
    ckpt_path=ckpt_attn_s2,
    device=device,
    train_config=attn_config,
    skip_training=ckpt_attn_s2.exists() and not skip_attn_s2
)

## Compare Training Results (Stage 2)

Side-by-side comparison of Standard U-Net vs. UNetAttention training curves. We look for faster convergence or lower final validation loss to see if the attention mechanism helps.

In [ ]:
if 'hist_unet_s2' not in locals() or not hist_unet_s2:
    ckpt_unet_s2 = CHECKPOINT_DIR / f"model_a_unet_stage2_{CHUNK_DURATION:.0f}s.pth"
    hist_unet_s2 = utils.load_training_history_from_checkpoint(ckpt_unet_s2)

if 'hist_attn_s2' not in locals() or not hist_attn_s2:
    ckpt_attn_s2 = CHECKPOINT_DIR / f"model_unetattention_stage2_{CHUNK_DURATION:.0f}s.pth"
    hist_attn_s2 = utils.load_training_history_from_checkpoint(ckpt_attn_s2)

test_results_ckpt_unet_s2 = CHECKPOINT_DIR / f"test_results_unet_stage2_{CHUNK_DURATION:.0f}s.pkl"
test_results_ckpt_attn_s2 = CHECKPOINT_DIR / f"test_results_unetattention_stage2_{CHUNK_DURATION:.0f}s.pkl"

test_unet_s2 = utils.load_test_results(test_results_ckpt_unet_s2)
test_attn_s2 = utils.load_test_results(test_results_ckpt_attn_s2)

if hist_unet_s2 and hist_attn_s2:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    if 'train_loss' in hist_unet_s2:
        epochs_unet = range(1, len(hist_unet_s2['train_loss']) + 1)
        ax1.plot(epochs_unet, hist_unet_s2['train_loss'], 'o-', label='Train', linewidth=2)
        ax1.plot(epochs_unet, hist_unet_s2['val_loss'], 's--', label='Val', linewidth=2)

        if test_unet_s2 and 'mean' in test_unet_s2 and 'std' in test_unet_s2:
            ax1.fill_between(
                epochs_unet,
                test_unet_s2['mean'] - test_unet_s2['std'],
                test_unet_s2['mean'] + test_unet_s2['std'],
                alpha=0.2, color='red'
            )
            ax1.axhline(
                test_unet_s2['mean'],
                color='red', linestyle=':', linewidth=2,
                label=f"Test (μ={test_unet_s2['mean']:.4f})"
            )

    ax1.set_title('Standard U-Net - Stage 2', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    if 'train_loss' in hist_attn_s2:
        epochs_attn = range(1, len(hist_attn_s2['train_loss']) + 1)
        ax2.plot(epochs_attn, hist_attn_s2['train_loss'], 'o-', label='Train', linewidth=2)
        ax2.plot(epochs_attn, hist_attn_s2['val_loss'], 's--', label='Val', linewidth=2)

        if test_attn_s2 and 'mean' in test_attn_s2 and 'std' in test_attn_s2:
            ax2.fill_between(
                epochs_attn,
                test_attn_s2['mean'] - test_attn_s2['std'],
                test_attn_s2['mean'] + test_attn_s2['std'],
                alpha=0.2, color='red'
            )
            ax2.axhline(
                test_attn_s2['mean'],
                color='red', linestyle=':', linewidth=2,
                label=f"Test (μ={test_attn_s2['mean']:.4f})"
            )

    ax2.set_title('UNetAttention - Stage 2', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.suptitle('Train/Val/Test Comparison - Stage 2 (U-Net vs UNetAttention)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Could not plot comparison - missing training history for one or both models.")

## Stage 2 Visual Evaluation (Spectrograms + Audio)

Visual inspection of the Attention model's separation quality on a test sample. We look for clearer separation in the spectrograms compared to previous models.

In [ ]:
print("="*70)
print("STAGE 2 EVALUATION: U-Net vs UNetAttention (Full Band Mixture → Vocals)")
print("="*70)

model_unet = model_unet.to(device)
model_attn_s2 = model_attn_s2.to(device)
model_unet.eval()
model_attn_s2.eval()

SR = 22050
DURATION = 120.0
CHUNK_LEN = 8.0

test_base = DATA_DIR / 'stage2' / 'test'
print(f"\nSearching for Stage 2 test data at: {test_base}")

if (test_base / 'mixture').exists():
    mix_dir = test_base / 'mixture'
    print("Found subdirectories: mixture/ and target/")
else:
    print("ERROR: Stage 2 Test directory not found!")
    raise FileNotFoundError(f"Test data directory not found: {test_base}")

all_mix_files = sorted(list(mix_dir.glob('*.npy')))
if len(all_mix_files) == 0:
    raise FileNotFoundError("No .npy files found")

songs = {}
for f in all_mix_files:
    name = f.stem
    if '_chunk' in name:
        parts = name.rsplit('_chunk', 1)
        if len(parts) == 2 and parts[1].isdigit():
            song_name = parts[0]
            idx = int(parts[1])
            if song_name not in songs:
                songs[song_name] = {}
            songs[song_name][idx] = f
    else:
        parts = name.split('_')
        if len(parts) >= 2 and parts[-1].isdigit():
            idx = int(parts[-1])
            song_name = '_'.join(parts[:-1])
            if song_name not in songs:
                songs[song_name] = {}
            songs[song_name][idx] = f

song_list = sorted(songs.keys())

if len(song_list) == 1:
    selected = song_list[0]
    print(f"\nAuto-selected: {selected} (only available song)")
else:
    print(f"\nSelect a song [0-{len(song_list)-1}] or press Enter for default (0):")
    try:
        choice = input("Choice: ").strip()
        idx = int(choice) if choice else 0
    except:
        idx = 0
    selected = song_list[idx]
    print(f"Selected: {selected}")

chunks = songs[selected]

print("Stitching chunks...")
HOP_LENGTH = 4.0
hop_samples = int(HOP_LENGTH * SR)
target_samples = int(DURATION * SR)
mix_wav = np.zeros(target_samples, dtype=np.float32)
tgt_wav = np.zeros(target_samples, dtype=np.float32)
weights = np.zeros(target_samples, dtype=np.float32)
has_target = True

for i in sorted(chunks.keys()):
    pos = i * hop_samples
    if pos >= target_samples:
        break

    mix_chunk = np.load(chunks[i])
    tgt_file = chunks[i].name.replace('mix_', 'tgt_')
    tgt_path = test_base / 'target' / tgt_file

    if tgt_path.exists():
        tgt_chunk = np.load(tgt_path)
    else:
        tgt_chunk = np.zeros_like(mix_chunk)
        has_target = False

    valid_len = min(len(mix_chunk), target_samples - pos)
    window = np.hanning(len(mix_chunk))[:valid_len]

    mix_wav[pos:pos+valid_len] += mix_chunk[:valid_len] * window
    tgt_wav[pos:pos+valid_len] += tgt_chunk[:valid_len] * window
    weights[pos:pos+valid_len] += window

mix_wav = np.divide(mix_wav, weights, where=weights > 0)
tgt_wav = np.divide(tgt_wav, weights, where=weights > 0)

print(f"Stitching complete: mix_wav shape={mix_wav.shape}, range=[{mix_wav.min():.4f}, {mix_wav.max():.4f}]")
print(f"Stitching complete: tgt_wav shape={tgt_wav.shape}, range=[{tgt_wav.min():.4f}, {tgt_wav.max():.4f}]")

print("\nRunning Stage 2 inference...")
est_unet_s2 = utils.sliding_window_inference(
    model_unet, processor_unet, mix_wav, chunk_len=CHUNK_LEN, sr=SR, device=device
)
est_attn_s2 = utils.sliding_window_inference(
    model_attn_s2, processor_attn_s2, mix_wav, chunk_len=CHUNK_LEN, sr=SR, device=device
)

print(f"\n{'='*70}\nSTAGE 2 RESULTS: U-Net vs UNetAttention\n{'='*70}")

fig, axes = plt.subplots(3, 2, figsize=(15, 12))

axes[0,0].imshow(utils.to_spec(mix_wav, processor_unet), aspect='auto', origin='lower', cmap='viridis')
axes[0,0].set_title("Input: Full Band Mixture", fontweight='bold')

if has_target:
    axes[0,1].imshow(utils.to_spec(tgt_wav, processor_unet), aspect='auto', origin='lower', cmap='viridis')
    axes[0,1].set_title("Ground Truth: Vocals", fontweight='bold')
else:
    axes[0,1].text(0.5, 0.5, "Target Not Available", ha='center', va='center', transform=axes[0,1].transAxes)

spec_tgt_unet = utils.to_spec(tgt_wav, processor_unet)
spec_pred_unet = utils.to_spec(est_unet_s2, processor_unet)
min_len_u = min(spec_tgt_unet.shape[1], spec_pred_unet.shape[1])
err_unet = np.abs(spec_pred_unet[:, :min_len_u] - spec_tgt_unet[:, :min_len_u])

axes[1,0].imshow(err_unet, aspect='auto', origin='lower', cmap='magma')
axes[1,0].set_title("U-Net Error Map (|Pred - Tgt|)", fontweight='bold')

axes[1,1].imshow(spec_pred_unet, aspect='auto', origin='lower', cmap='viridis')
axes[1,1].set_title("U-Net Prediction (Stage 2)", fontweight='bold')

spec_tgt_attn = utils.to_spec(tgt_wav, processor_attn_s2)
spec_pred_attn = utils.to_spec(est_attn_s2, processor_attn_s2)
min_len_a = min(spec_tgt_attn.shape[1], spec_pred_attn.shape[1])
err_attn = np.abs(spec_pred_attn[:, :min_len_a] - spec_tgt_attn[:, :min_len_a])

axes[2,0].imshow(err_attn, aspect='auto', origin='lower', cmap='magma')
axes[2,0].set_title("UNetAttention Error Map (|Pred - Tgt|)", fontweight='bold')

axes[2,1].imshow(spec_pred_attn, aspect='auto', origin='lower', cmap='viridis')
axes[2,1].set_title("UNetAttention Prediction (Stage 2)", fontweight='bold')

for ax in axes.flatten():
    ax.set_xlabel("Time")
    ax.set_ylabel("Frequency")

plt.suptitle(f"Stage 2 Evaluation: {selected}", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nAudio Playback:")
print("Input (Mix):")
display(Audio(mix_wav, rate=SR))

if has_target:
    print("\nGround Truth (Target):")
    display(Audio(tgt_wav, rate=SR))

print("\nU-Net Prediction:")
display(Audio(est_unet_s2, rate=SR))

print("\nUNetAttention Prediction:")
display(Audio(est_attn_s2, rate=SR))

## Upload Your Own Song for Instrumental Extraction

In [ ]:
UPLOAD_DIR = DATA_DIR / "user_uploads"
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    try:
        from google.colab import files
        uploaded = files.upload()
        for name, data in uploaded.items():
            out_path = UPLOAD_DIR / name
            with open(out_path, "wb") as f:
                f.write(data)
            print(f"Saved: {out_path}")
    except Exception as e:
        print(f"Colab upload failed: {e}")
        print(f"Place a file manually in: {UPLOAD_DIR}")
else:
    print(f"Place your audio file in: {UPLOAD_DIR}")

exts = ["*.wav", "*.mp3", "*.flac", "*.ogg", "*.m4a"]
audio_files = []
for ext in exts:
    audio_files += list(UPLOAD_DIR.glob(ext))

audio_files = sorted(audio_files, key=lambda p: p.stat().st_mtime, reverse=True)

if not audio_files:
    print("No audio files found in upload folder.")
else:
    audio_path = audio_files[0]
    print(f"Using file: {audio_path.name}")
    
    import librosa
    audio, sr = librosa.load(str(audio_path), sr=22050, mono=True)
    print(f"Loaded audio: {len(audio)/sr:.2f} seconds")
    
    ckpt_unet_s2 = CHECKPOINT_DIR / f"model_a_unet_stage2_{CHUNK_DURATION:.0f}s.pth"
    if ckpt_unet_s2.exists():
        print(f"\nLoading U-Net Stage 2 weights from: {ckpt_unet_s2.name}")
        checkpoint = torch.load(ckpt_unet_s2, map_location=device)
        model_unet.load_state_dict(checkpoint['model_state_dict'])
        print("U-Net Stage 2 weights loaded")
    else:
        print(f"Warning: U-Net Stage 2 checkpoint not found at {ckpt_unet_s2}")
    
    print("\nRunning inference with both models...")
    print("   - Standard U-Net (Stage 2)")
    unet_output = utils.sliding_window_inference(
        model_unet,
        processor_unet,
        audio,
        chunk_len=8.0,
        sr=22050,
        device=device
    )
    
    print("   - UNetAttention (Stage 2)")
    unetattention_output = utils.sliding_window_inference(
        model_attn_s2,
        processor_attn_s2,
        audio,
        chunk_len=8.0,
        sr=22050,
        device=device
    )
    
    print(f"\n{'='*70}")
    print("MODEL COMPARISON: U-Net vs UNetAttention (Stage 2)")
    print(f"{'='*70}\n")
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 12))
    
    axes[0].imshow(utils.to_spec(audio, processor_unet), aspect='auto', origin='lower', cmap='viridis')
    axes[0].set_title("Original Mix", fontweight='bold')
    axes[0].set_ylabel("Frequency")
    axes[0].set_xlabel("Time")
    
    axes[1].imshow(utils.to_spec(unet_output, processor_unet), aspect='auto', origin='lower', cmap='viridis')
    axes[1].set_title("U-Net Output (Stage 2)", fontweight='bold')
    axes[1].set_ylabel("Frequency")
    axes[1].set_xlabel("Time")
    
    axes[2].imshow(utils.to_spec(unetattention_output, processor_attn_s2), aspect='auto', origin='lower', cmap='viridis')
    axes[2].set_title("UNetAttention Output (Stage 2)", fontweight='bold')
    axes[2].set_ylabel("Frequency")
    axes[2].set_xlabel("Time")
    
    plt.tight_layout()
    plt.show()
    
    print("Audio results:\n")
    print("Original Mix:")
    display(Audio(audio, rate=22050))
    
    print("\nU-Net Output:")
    display(Audio(unet_output, rate=22050))
    
    print("\nUNetAttention Output:")
    display(Audio(unetattention_output, rate=22050))
    
    output_unet = UPLOAD_DIR / f"{audio_path.stem}_unet_stage2.wav"
    output_unetattention = UPLOAD_DIR / f"{audio_path.stem}_unetattention_stage2.wav"
    
    import soundfile as sf
    sf.write(output_unet, unet_output, 22050)
    sf.write(output_unetattention, unetattention_output, 22050)
    
    print(f"\nSaved outputs:")
    print(f"   U-Net: {output_unet.name}")
    print(f"   UNetAttention: {output_unetattention.name}")

## Quantitative Evaluation

Compute BSS metrics (SDR/SIR/SAR) on test set using museval library.

In [ ]:
NUM_TEST_SAMPLES = 500
STAGE_FOR_EVAL = "stage2"
RANDOM_SAMPLE_TEST_CHUNKS = True
RANDOM_SEED = 42
FORCE_RECOMPUTE_METRICS = False

metrics_ckpt = CHECKPOINT_DIR / f"quant_metrics_unet_vs_unetattention_{STAGE_FOR_EVAL}_{NUM_TEST_SAMPLES}samples.pkl"
print(f"Metrics checkpoint: {metrics_ckpt}")

if FORCE_RECOMPUTE_METRICS and metrics_ckpt.exists():
    metrics_ckpt.unlink()
    print("Removed existing metrics checkpoint (force recompute enabled).")


metrics_raw = utils.evaluate_separation_quality(
    model_1=model_unet,
    model_2=model_attn_s2,
    processor_1=processor_unet,
    processor_2=processor_attn_s2,
    test_data_dir=DATA_DIR,
    stage=STAGE_FOR_EVAL,
    num_samples=NUM_TEST_SAMPLES,
    sr=22050,
    device=device,
    save_path=metrics_ckpt,
    load_if_exists=True,
    random_sampling=RANDOM_SAMPLE_TEST_CHUNKS,
    random_seed=RANDOM_SEED
)

# Training a unettention model to extract vocals only

In [ ]:
print(f"\n{'='*70}")
print('TRAINING: UNetAttention (Stage 2 Mix -> Clean Vocals)')
print(f"{'='*70}\n")

train_mix_dir = DATA_DIR / 'stage2' / 'train' / 'mixture'
train_voc_dir = DATA_DIR / 'vocals' / 'train'
val_mix_dir = DATA_DIR / 'stage2' / 'val' / 'mixture'
val_voc_dir = DATA_DIR / 'vocals' / 'val'

ckpt_voc_attn = CHECKPOINT_DIR / 'model_unetattention_vocals_only.pth'

def get_paired_files(mix_dir, voc_dir, split_name):
    mix_files = sorted(list(mix_dir.glob('*.npy')))
    voc_files_dict = {f.name: f for f in voc_dir.glob('*.npy')}

    valid_pairs = [(mf, voc_files_dict[mf.name]) for mf in mix_files if mf.name in voc_files_dict]
    print(f"   {split_name}: Found {len(valid_pairs)} pairs")

    if not valid_pairs:
        return [], []

    mix_list = [p[0] for p in valid_pairs]
    tgt_list = [p[1] for p in valid_pairs]
    return mix_list, tgt_list

if not train_mix_dir.exists() or not train_voc_dir.exists():
    print("Training folders not found (data_sub mode detected).")
    if ckpt_voc_attn.exists():
        print(f"Using existing checkpoint: {ckpt_voc_attn.name}")
    else:
        print("No checkpoint found, and training data is unavailable.")
        print("Provide full data (train/val) or place a pretrained checkpoint in checkpoints/.")
else:
    print("Matching files...")
    train_mix, train_tgt = get_paired_files(train_mix_dir, train_voc_dir, "Train")

    if val_mix_dir.exists() and val_voc_dir.exists():
        val_mix, val_tgt = get_paired_files(val_mix_dir, val_voc_dir, "Validation")
    else:
        print("Validation folders not found. Reusing train pairs for validation.")
        val_mix, val_tgt = train_mix, train_tgt

    if not train_mix:
        print("No training pairs found.")
        if ckpt_voc_attn.exists():
            print(f"Using existing checkpoint: {ckpt_voc_attn.name}")
        else:
            raise RuntimeError("Cannot train: no paired training files and no existing checkpoint.")
    else:
        train_dataset = utils.StandardDataset(train_mix, train_tgt)
        val_dataset = utils.StandardDataset(val_mix, val_tgt)

        train_loader = DataLoader(
            train_dataset,
            batch_size=4,
            shuffle=True,
            num_workers=2 if torch.cuda.is_available() else 0,
            pin_memory=torch.cuda.is_available()
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=4,
            shuffle=False,
            num_workers=2 if torch.cuda.is_available() else 0,
            pin_memory=torch.cuda.is_available()
        )

        model_voc_attn = ma.UNetAttention(
            in_channels=1,
            out_channels=1,
            base_filters=32,
            num_layers=5,
            num_heads=4,
            batchnorm=True,
            dropout=0.1
        ).to(device)

        processor_voc = utils.AudioProcessor(device=device)
        optimizer_voc = optim.Adam(model_voc_attn.parameters(), lr=1e-4)
        loss_fn_voc = nn.MSELoss()

        voc_trainer = utils.UniversalTrainer(
            model=model_voc_attn,
            train_loader=train_loader,
            val_loader=val_loader,
            processor=processor_voc,
            optimizer=optimizer_voc,
            loss_fn=loss_fn_voc,
            device=device,
            input_type='spectrogram'
        )

        if ckpt_voc_attn.exists():
            print(f"Checkpoint found: {ckpt_voc_attn.name}")
        else:
            print("Starting Vocal Extraction Training...")
            voc_history = voc_trainer.train(num_epochs=20, save_path=ckpt_voc_attn)

            plt.figure(figsize=(10, 5))
            plt.plot(voc_history['train_loss'], label='Train Loss')
            plt.plot(voc_history['val_loss'], label='Val Loss')
            plt.title("UNetAttention: Vocal Extraction Training")
            plt.xlabel("Epoch")
            plt.ylabel("MSE Loss")
            plt.legend()
            plt.show()

## Verify Training Success - UNetAttention

In [ ]:
print("="*70)
print("VOCAL EXTRACTION FROM TEST SET (Stage 2 - UNetAttention)")
print("="*70)

if 'model_voc_attn' not in locals():
    model_voc_attn = ma.UNetAttention(
        in_channels=1,
        out_channels=1,
        base_filters=32,
        num_layers=5,
        num_heads=4,
        batchnorm=True,
        dropout=0.1
    ).to(device)

if 'processor_voc' not in locals():
    processor_voc = utils.AudioProcessor(device=device)

ckpt_voc_attn = CHECKPOINT_DIR / 'model_unetattention_vocals_only.pth'
if ckpt_voc_attn.exists():
    checkpoint = torch.load(ckpt_voc_attn, map_location=device, weights_only=False)
    model_voc_attn.load_state_dict(checkpoint['model_state_dict'])
    model_voc_attn.eval()
    print(f"Model loaded: {ckpt_voc_attn.name}")
else:
    print(f"Model not found: {ckpt_voc_attn}")
    raise FileNotFoundError(f"Checkpoint {ckpt_voc_attn} does not exist")

DURATION = 120.0
CHUNK_LEN = 8.0
HOP_LENGTH = 4.0

mix_base = DATA_DIR / 'stage2' / 'test'
vocal_base = DATA_DIR / 'vocals' / 'test'
mix_dir = mix_base / 'mixture'
vocal_dir = vocal_base

if not mix_dir.exists() or not vocal_dir.exists():
    print("ERROR: Test directories not found")
    print(f"   Mixture dir: {mix_dir.exists()}")
    print(f"   Vocals dir: {vocal_dir.exists()}")
    raise FileNotFoundError("Test data directories not found")

print("Found test directories")

def parse_song_and_idx(stem):
    if '_chunk' in stem:
        parts = stem.rsplit('_chunk', 1)
        if len(parts) == 2 and parts[1].isdigit():
            return parts[0], int(parts[1])

    parts = stem.split('_')
    if len(parts) >= 2 and parts[-1].isdigit():
        return '_'.join(parts[:-1]), int(parts[-1])

    return stem, 0

all_mix_files = sorted(list(mix_dir.glob('*.npy')))
all_vocal_files = sorted(list(vocal_dir.glob('*.npy')))

if len(all_mix_files) == 0:
    raise FileNotFoundError("No mixture files found")
if len(all_vocal_files) == 0:
    raise FileNotFoundError("No vocal files found")

songs = {}
for f in all_mix_files:
    song_name, idx = parse_song_and_idx(f.stem)
    if song_name not in songs:
        songs[song_name] = {}
    songs[song_name][idx] = f

vocal_song_names = set(parse_song_and_idx(f.stem)[0] for f in all_vocal_files)
song_list = sorted([s for s in songs.keys() if s in vocal_song_names])

if not song_list:
    song_list = sorted(songs.keys())

if len(song_list) == 1:
    selected_song = song_list[0]
    print(f"\nAuto-selected: {selected_song} (only available song)")
else:
    print(f"\nFound {len(song_list)} songs:")
    for i, song in enumerate(song_list):
        print(f"   {i}: {song}")

    print(f"\nSelect a song [0-{len(song_list)-1}] or press Enter for default (0):")
    try:
        choice = input("Choice: ").strip()
        idx = int(choice) if choice else 0
        if idx < 0 or idx >= len(song_list):
            idx = 0
    except Exception:
        idx = 0
    selected_song = song_list[idx]

print(f"\nSelected: {selected_song}")

chunks = songs[selected_song]
print(f"   Found {len(chunks)} chunks")

print("\nStitching chunks...")
hop_samples = int(HOP_LENGTH * SR)
target_samples = int(DURATION * SR)
mix_wav = np.zeros(target_samples, dtype=np.float32)
vocal_wav_gt = np.zeros(target_samples, dtype=np.float32)
weights = np.zeros(target_samples, dtype=np.float32)
has_target = True

for chunk_idx in sorted(chunks.keys()):
    pos = chunk_idx * hop_samples
    if pos >= target_samples:
        break

    mix_chunk = np.load(chunks[chunk_idx])

    vocal_filename = chunks[chunk_idx].name
    vocal_path = vocal_dir / vocal_filename

    if vocal_path.exists():
        vocal_chunk = np.load(vocal_path)
    else:
        vocal_chunk = np.zeros_like(mix_chunk)
        has_target = False

    valid_len = min(len(mix_chunk), target_samples - pos)
    window = np.hanning(len(mix_chunk))[:valid_len]

    mix_wav[pos:pos+valid_len] += mix_chunk[:valid_len] * window
    vocal_wav_gt[pos:pos+valid_len] += vocal_chunk[:valid_len] * window
    weights[pos:pos+valid_len] += window

mask = weights > 0
mix_wav[mask] = mix_wav[mask] / weights[mask]
vocal_wav_gt[mask] = vocal_wav_gt[mask] / weights[mask]

print("Stitching complete:")
print(f"   Mix shape: {mix_wav.shape}, range=[{mix_wav.min():.4f}, {mix_wav.max():.4f}]")
print(f"   Vocal GT shape: {vocal_wav_gt.shape}, range=[{vocal_wav_gt.min():.4f}, {vocal_wav_gt.max():.4f}]")

print("\nRunning UNetAttention inference...")
model_voc_attn.to(device)
model_voc_attn.eval()
extracted_vocals = utils.sliding_window_inference(
    model_voc_attn, processor_voc, mix_wav,
    chunk_len=CHUNK_LEN, sr=SR, device=device
)

print("Inference complete:")
print(f"   Extracted shape: {extracted_vocals.shape}")

print(f"\n{'='*70}\nVISUALIZATION\n{'='*70}")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

spec_mix = utils.to_spec(mix_wav, processor_voc)
im0 = axes[0, 0].imshow(spec_mix, aspect='auto', origin='lower', cmap='viridis')
axes[0, 0].set_title("Input: Full Band Mixture", fontweight='bold', fontsize=12)
plt.colorbar(im0, ax=axes[0, 0])

if has_target:
    spec_vocal_gt = utils.to_spec(vocal_wav_gt, processor_voc)
    im1 = axes[0, 1].imshow(spec_vocal_gt, aspect='auto', origin='lower', cmap='viridis')
    axes[0, 1].set_title("Ground Truth: Vocals", fontweight='bold', fontsize=12)
    plt.colorbar(im1, ax=axes[0, 1])
else:
    axes[0, 1].text(0.5, 0.5, "Target Not Available", ha='center', fontsize=14)
    axes[0, 1].set_title("Ground Truth: Vocals", fontweight='bold', fontsize=12)

spec_pred = utils.to_spec(extracted_vocals, processor_voc)
im2 = axes[1, 0].imshow(spec_pred, aspect='auto', origin='lower', cmap='viridis')
axes[1, 0].set_title("Extracted Vocals (UNetAttention)", fontweight='bold', fontsize=12)
plt.colorbar(im2, ax=axes[1, 0])

if has_target:
    min_len = min(spec_vocal_gt.shape[1], spec_pred.shape[1])
    err = np.abs(spec_pred[:, :min_len] - spec_vocal_gt[:, :min_len])
    im3 = axes[1, 1].imshow(err, aspect='auto', origin='lower', cmap='hot')
    axes[1, 1].set_title("Error Map |Pred - GT|", fontweight='bold', fontsize=12)
    plt.colorbar(im3, ax=axes[1, 1])
    mse = np.mean(err**2)
    print(f"   MSE Loss: {mse:.6f}")
else:
    axes[1, 1].text(0.5, 0.5, "No Target Available\nfor Error Computation", ha='center', fontsize=12)
    axes[1, 1].set_title("Error Map", fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

print("\nAudio Comparison:")
print("   1) Input Mixture:")
display(Audio(mix_wav, rate=SR))

if has_target:
    print("\n   2) Ground Truth Vocals:")
    display(Audio(vocal_wav_gt, rate=SR))

print("\n   3) Extracted Vocals (UNetAttention):")
display(Audio(extracted_vocals, rate=SR))

print("\nExtraction complete!")